In [22]:
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter

from langchain.storage import LocalFileStore
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.faiss import FAISS

from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough
from langchain.memory import ConversationBufferMemory

class BufferMemory:
    MEMORY_KEY = 'history'  # 메모리와 관련된 템플릿 키 값 

    def __init__(self):
        # 메모리 설정
        self.memory = ConversationBufferMemory(
            return_messages = True,
            memory_key = self.MEMORY_KEY,
        )

    # 프롬프트 템플릿 사용 용
    def prompt_template(self):
        return MessagesPlaceholder(variable_name = self.MEMORY_KEY)
    
    # 메모리에 기록 된 내역 가져오기
    def load_memory(self, _):
        return self.memory.load_memory_variables({})[self.MEMORY_KEY]
    
    # 메모리에 기록 된 내역 가져오기
    def save_memory(self, input, output):
        return self.memory.save_context({"input": input}, {"output": output})

# 모델 설정
llm = ChatOpenAI(temperature=0.1)

# 캐시 경로 설정
cache_dir = LocalFileStore("./.cache/")

# 문서 불러오기
docs = UnstructuredFileLoader("./files/document.txt").load()

# 문서 잘라낼 속성
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator = "\n",       # 잘라낼 구간 설정
    chunk_size = 400,       # 잘라낼 텍스트 크기
    chunk_overlap = 100,    # 잘라낼 때 앞부분 곂치는 문장을 얼만큼 더 가져올 건지
)
# 문서 자르기
splitDocs = splitter.split_documents(docs)

# 임베딩 캐시 설정 (캐싱 된 벡터는 다시 만들지 않기 위함)
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings = OpenAIEmbeddings(),     # 임베딩 모델 (OpenAI)
    document_embedding_cache = cache_dir,           # 캐싱될 문서 경로
)

# 자른 문서 기준으로 벡터 생성 (Store, 임베딩 모델에 의해서 캐시 처리된 백터는 재활용 함)
vectorStore = FAISS.from_documents(splitDocs, cached_embeddings)

# 리트리버 설정 (벡터 Store)
retriever = vectorStore.as_retriever()

# 메모리 설정
memory = BufferMemory()

# 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 훌륭한 조수 입니다. 주어진 문맥만을 이용하여 질문에 대답하세요. 답을 모른다면, 모른다고 하세요. 모르는 정보를 지어내지 마세요:\n\n{context}"),
    memory.prompt_template(),
    ("human", "{question}"),
])

# 문맥 가져오기
def get_context(query):
    # 리트리버가 골라낸 문장 가져오기
    docs = retriever.get_relevant_documents(query)

    # 각 문서의 page_content를 합쳐 하나의 문자열로 반환
    return "\n".join([doc.page_content for doc in docs])

# 체인 설정
chain = {
    "context": get_context,             # 리트리버가 골라낸 문장 context에 설정
    "question": RunnablePassthrough(),  # 아래 질문 PassThrough
} | RunnablePassthrough.assign(history=memory.load_memory) | prompt | llm


# 체인 구동 함수 설정 
def chain_invoke(question):
    result = chain.invoke(question).content     # 질문 결과
    memory.save_memory(question, result)        # 메모리 기록
    return result


In [23]:
chain_invoke("Aaronson 은 유죄인가요?")

'네, Jones, Aaronson, 그리고 Rutherford은 그들이 기소된 범죄로 유죄 판결을 받았습니다.'

In [24]:
chain_invoke("그가 테이블에 어떤 메시지를 썼나요?")

'그가 쓴 메시지는 "자유는 노예다"와 "2와 2는 5다" 입니다.'

In [25]:
chain_invoke("Julia 는 누구인가요?")

'Julia는 윈스턴과 사랑에 빠진 여성으로, 이야기에서 중요한 역할을 합니다.'

In [26]:
memory.load_memory('')

[HumanMessage(content='Aaronson 은 유죄인가요?'),
 AIMessage(content='네, Jones, Aaronson, 그리고 Rutherford은 그들이 기소된 범죄로 유죄 판결을 받았습니다.'),
 HumanMessage(content='그가 테이블에 어떤 메시지를 썼나요?'),
 AIMessage(content='그가 쓴 메시지는 "자유는 노예다"와 "2와 2는 5다" 입니다.'),
 HumanMessage(content='Julia 는 누구인가요?'),
 AIMessage(content='Julia는 윈스턴과 사랑에 빠진 여성으로, 이야기에서 중요한 역할을 합니다.')]